# Day 13: Pandas 基础 —— 习题

> **范围**: DataFrame 创建、属性、loc/iloc、筛选、赋值、排序、类型转换、日期处理
> **数据**: `../data/sales.csv`（500行，9列）
> **建议用时**: 60-90 分钟
> **提示**: Pandas 和 NumPy 一样，能用向量化就不用循环

## Easy

**1. 读取与基本属性**

用 `pd.read_csv` 读取 `../data/sales.csv`，然后完成：
- 打印 DataFrame 的形状（shape）
- 打印所有列名（columns）
- 打印每列的数据类型（dtypes）
- 查看前 5 行和后 5 行
- 对数值列调用 `describe()`，打印结果
- 统计每列的非空值数量（用 `.count()`）

In [3]:
import pandas as pd

df = pd.read_csv("../data/sales.csv")

print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.head(5))
print(df.tail(5))
print(df.describe())
print(df.count())

(500, 9)
Index(['order_id', 'customer_id', 'product', 'category', 'quantity', 'price',
       'order_date', 'country', 'total'],
      dtype='object')
order_id       object
customer_id    object
product        object
category       object
quantity        int64
price           int64
order_date     object
country        object
total           int64
dtype: object
  order_id customer_id     product   category  quantity  price  order_date  \
0    O1000        C007    Keyboard  Accessory         2   1299  2024-01-01   
1    O1001        C004    Keyboard  Accessory         1     99  2024-01-01   
2    O1002        C005      Laptop   Computer         4     99  2024-01-02   
3    O1003        C007  Headphones      Audio         4     99  2024-01-03   
4    O1004        C003       Phone     Mobile         5     99  2024-01-03   

   country  total  
0  Germany   2598  
1       US     99  
2       US    396  
3       US    396  
4   France    495  
    order_id customer_id   product   category  q

**2. loc 与 iloc 取行列**

基于上面的 `df`：
- 用 `loc` 取出第 10 行（索引 10）的 `order_id` 和 `total` 两个值
- 用 `loc` 取出索引 0~5（含5）的所有行的 `order_id`、`customer_id`、`country` 三列
- 用 `iloc` 取出前 5 行的前 3 列（order_id, customer_id, product）
- 用 `at` 取第 100 行的 `total` 值
- 用 `iat` 取第 200 行第 3 列（product）的值

In [7]:
print(df.loc[10,['order_id', 'total']])
print(df.loc[0:5, ['order_id', 'customer_id', 'country']])
print(df.iloc[0:5, 0:3])
print(df.at[100, 'total'])
print(df.iat[199, 2])


order_id    O1010
total        1797
Name: 10, dtype: object
  order_id customer_id  country
0    O1000        C007  Germany
1    O1001        C004       US
2    O1002        C005       US
3    O1003        C007       US
4    O1004        C003   France
5    O1005        C008       UK
  order_id customer_id     product
0    O1000        C007    Keyboard
1    O1001        C004    Keyboard
2    O1002        C005      Laptop
3    O1003        C007  Headphones
4    O1004        C003       Phone
5196
Keyboard


**3. 布尔筛选基础**

基于 `df`：
- 筛选出 `total` 大于 **所有订单平均值** 的订单，返回一个 DataFrame
- 统计这个 DataFrame 有多少行（len 或 .shape）
- 筛选出 `country` 为 `"UK"` 的所有订单，查看前3行
- 筛选出 `category` 为 `"Computer"` 且 `total` > 3000 的订单
- 用 `isin` 筛选出 `country` 在 `["US", "France", "Germany"]` 中的订单，统计数量

In [14]:
mean = df['total'].mean()
sub_df = df[df['total'] > mean]
print(sub_df.shape)
print(sub_df[sub_df['country'] == 'UK'].head(3))
print(sub_df[(sub_df['category'] == 'Computer') & (sub_df['total'] > 3000)])
print(len(sub_df['country'].isin(['US', 'France', 'Germany'])))

(189, 9)
   order_id customer_id   product   category  quantity  price  order_date  \
6     O1006        C005     Phone     Mobile         5   1299  2024-01-05   
23    O1023        C008  Keyboard  Accessory         5    999  2024-01-17   
27    O1027        C002     Phone     Mobile         5   1299  2024-01-20   

   country  total  
6       UK   6495  
23      UK   4995  
27      UK   6495  
    order_id customer_id  product  category  quantity  price  order_date  \
18     O1018        C008   Laptop  Computer         3   1999  2024-01-14   
32     O1032        C004   Laptop  Computer         5    999  2024-01-24   
37     O1037        C001  Monitor  Computer         4   1299  2024-01-28   
38     O1038        C001  Monitor  Computer         2   1999  2024-01-28   
99     O1099        C005  Monitor  Computer         5   1299  2024-03-13   
100    O1100        C007   Laptop  Computer         4   1299  2024-03-14   
106    O1106        C006  Monitor  Computer         5   1299  2024-03-

参考答案

问题: sub_df['country'].isin([...]) 返回的是布尔 Series（[True, False, True...]），len() 返回的是 Series 的长度（189），而不是 True 的数量。

正确做法:

In [40]:
# 方法1: 布尔 Series 的 sum() = True 的数量
print(sub_df['country'].isin(['US', 'France', 'Germany']).sum())

# 方法2: 先筛选再 len
print(len(sub_df[sub_df['country'].isin(['US', 'France', 'Germany'])]))

0
0


为什么错: 和 Day 11 题6 的「占比」错误类似——混淆了「数组长度」和「满足条件的数量」。isin 返回的是布尔面具，需要再套一层筛选或 .sum() 才能计数。

## Medium

**4. 新增列与筛选**

基于 `df`：
- 新增一列 `unit_price` = `total / quantity`（向量化，不用循环）
- 筛选出 `unit_price` > 500 的订单，查看 `order_id`、`product`、`unit_price` 前5行
- 新增一列 `price_level`：如果 `total` >= 3000 则为 `"高"`，否则为 `"低"`
  （提示：可用 `np.where(df["total"] >= 3000, "高", "低")`）
- 统计 `price_level` 列中 `"高"` 和 `"低"` 各有多少个（提示：`.value_counts()`）

In [18]:
import numpy as np

df['unit_price'] = df['total'] / df['quantity']
high_unit_price = df[df['unit_price'] > 500]
print(high_unit_price.loc[:,['order_id', 'product', 'unit_price']].head(5))
df['price_level'] = np.where(df['total'] >= 3000, '高', '低')
print(df['price_level'].value_counts())

   order_id   product  unit_price
0     O1000  Keyboard      1299.0
6     O1006     Phone      1299.0
7     O1007   Monitor       599.0
10    O1010   Monitor       599.0
11    O1011     Phone       599.0
price_level
低    362
高    138
Name: count, dtype: int64


**5. 修改值与删除列**

基于 `df`：
- 用 `loc` 把 `country` 为 `"US"` 的所有行的 `country` 改为 `"USA"`
- 验证修改后 `country` 列的唯一值（`.unique()`）
- 把 `customer_id` 为 `"C005"` 的所有订单的 `quantity` 增加 1（即 `quantity + 1`）
- 删除 `unit_price` 和 `price_level` 列（如果上面做过），创建新的 DataFrame `df_clean`
- 打印 `df_clean` 的列名，确认删除成功

In [21]:
df.loc[df['country'] == 'US', 'country'] = 'USA'
print(df['country'].unique())
df.loc[df['customer_id'] == 'C005', 'quantity'] += 1
df_clean = df.drop(columns=['unit_price', 'price_level'])
print(df_clean.columns)

['Germany' 'USA' 'France' 'UK' 'China']
Index(['order_id', 'customer_id', 'product', 'category', 'quantity', 'price',
       'order_date', 'country', 'total'],
      dtype='object')


**6. 排序、去重与重置索引**

基于 `df`：
- 按 `total` 降序排列，取出前 10 名订单的 `order_id` 和 `total`
- 先按 `country` 升序、再按 `total` 降序排列，查看前 10 行
- 对 `df` 按 `customer_id` 和 `country` 去重，比较去重前后的行数
- 筛选出 `country == "UK"` 的订单，重置索引（`drop=True`），验证新索引从 0 开始连续

In [24]:
df_sorted = df.sort_values('total', ascending=False)
print(df_sorted.head(10))
df_sorted2 = df.sort_values(['country', 'total'], ascending=[True, False])
print(df_sorted2.head(10))
df_nodup = df.drop_duplicates(subset=['customer_id', 'country'])
print(f"去重前: {len(df)}, 去重后: {len(df_nodup)}")
df_uk = df[df["country"] == "UK"].reset_index(drop=True)
print(df_uk.index.tolist())

    order_id customer_id     product   category  quantity  price  order_date  \
355    O1355        C007  Headphones      Audio         5   1999  2024-09-16   
370    O1370        C004       Phone     Mobile         5   1999  2024-09-27   
342    O1342        C003       Mouse  Accessory         5   1999  2024-09-07   
324    O1324        C006       Mouse  Accessory         5   1999  2024-08-24   
319    O1319        C006  Headphones      Audio         5   1999  2024-08-21   
432    O1432        C005    Keyboard  Accessory         7   1999  2024-11-11   
470    O1470        C005    Keyboard  Accessory         7   1999  2024-12-09   
33     O1033        C002       Mouse  Accessory         5   1999  2024-01-25   
409    O1409        C006       Phone     Mobile         5   1999  2024-10-26   
110    O1110        C008       Phone     Mobile         5   1999  2024-03-21   

     country  total  unit_price price_level  
355       UK   9995      1999.0           高  
370  Germany   9995      19

**7. 类型转换与缺失值**

基于 `df`：
- 把 `order_date` 列转换为 datetime 类型，存为新列 `order_dt`
- 把 `quantity` 列转换为整数类型（`astype(int)`）
- 把 `price` 列转换为浮点数（`pd.to_numeric`），查看是否有 NaN
- 检查 `df` 中每列的缺失值数量（`.isnull().sum()`）
- 从 `order_dt` 提取 `year` 和 `month` 两列

In [25]:
df['order_dt'] = pd.to_datetime(df['order_date'], errors="coerce")
df['quantity'] = df['quantity'].astype(int)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(df.isnull().sum())
print(df['order_dt'].dt.year)
print(df['order_dt'].dt.month)

order_id       0
customer_id    0
product        0
category       0
quantity       0
price          0
order_date     0
country        0
total          0
unit_price     0
price_level    0
order_dt       0
dtype: int64
0      2024
1      2024
2      2024
3      2024
4      2024
       ... 
495    2024
496    2024
497    2024
498    2024
499    2024
Name: order_dt, Length: 500, dtype: int32
0       1
1       1
2       1
3       1
4       1
       ..
495    12
496    12
497    12
498    12
499    12
Name: order_dt, Length: 500, dtype: int32


## Hard

**8. 日期分析 —— 按月份统计**

基于 `df`（已转换 `order_dt` 为 datetime）：
- 创建新列 `year_month` = `order_dt` 格式化为 `"YYYY-MM"` 字符串
  （提示：`df["order_dt"].dt.strftime("%Y-%m")`）
- 用 `value_counts()` 统计每个月份有多少订单
- 找出订单量最多的月份
- 计算每个月的 `total` 总和（提示：先筛选再 `.sum()` 或用 `groupby`——如果你学过）
  如果 groupby 还没学，可以用以下方式：
  ```python
  for month in df["year_month"].unique():
      month_df = df[df["year_month"] == month]
      print(month, month_df["total"].sum())
  ```

In [29]:
df['year_month'] = df["order_dt"].dt.strftime("%Y-%m")
print(df['year_month'].value_counts())

month_sum = df.groupby("year_month")["total"].sum()
print(month_sum)
top_month = month_sum.idxmax()
print(top_month)


year_month
2024-01    43
2024-07    43
2024-05    42
2024-03    42
2024-12    42
2024-10    42
2024-08    42
2024-04    41
2024-09    41
2024-06    41
2024-11    41
2024-02    40
Name: count, dtype: int64
year_month
2024-01     99365
2024-02     81795
2024-03    133371
2024-04    123882
2024-05     71176
2024-06    119061
2024-07    101271
2024-08    104388
2024-09    103183
2024-10    117265
2024-11    105984
2024-12    106975
Name: total, dtype: int64
2024-03


**9. 综合筛选 —— 多条件组合**

基于 `df`：
- 找出 **UK 的 Computer 品类** 且 **total > 2000** 的所有订单
- 找出 **US 或 France** 的订单中，`total` 在 **[1000, 3000]** 区间内（含边界）的订单
- 找出 `quantity` 为 **偶数** 且 `total` > 平均值的订单
  （提示：`df["quantity"] % 2 == 0`）
- 用 `loc` 取出上述结果中 `order_id`, `country`, `category`, `total` 四列，按 `total` 降序排列

In [36]:
print(df[(df['country'] == 'UK') & (df['product'] == 'Computer') & df['total'] > 2000].loc[:,['order_id', 'country', 'category', 'total']].sort_values('total', ascending = False))
print(df[(df['country'].isin(['USA', 'France'])) & (df['total'].between(1000, 3000))].loc[:,['order_id', 'country', 'category', 'total']].sort_values('total', ascending = False))
print(df[(df['quantity'] % 2 == 0) & (df['total'] > mean)].loc[:,['order_id', 'country', 'category', 'total']].sort_values('total', ascending = False))


Empty DataFrame
Columns: [order_id, country, category, total]
Index: []
    order_id country   category  total
14     O1014  France      Audio   2997
64     O1064     USA  Accessory   2997
307    O1307     USA      Audio   2997
264    O1264     USA      Audio   2997
148    O1148  France  Accessory   2997
..       ...     ...        ...    ...
154    O1154     USA  Accessory   1196
299    O1299     USA  Accessory   1196
404    O1404  France   Computer   1196
471    O1471  France   Computer   1196
454    O1454     USA   Computer   1196

[78 rows x 4 columns]
    order_id country   category  total
84     O1084      UK     Mobile   7996
92     O1092     USA  Accessory   7996
412    O1412      UK      Audio   7996
410    O1410      UK   Computer   7996
439    O1439      UK   Computer   7996
..       ...     ...        ...    ...
304    O1304  France     Mobile   2598
303    O1303   China     Mobile   2598
344    O1344      UK      Audio   2598
423    O1423     USA  Accessory   2598
465    O

参考答案

问题1（读题错误）: 题目要求的是 "Computer 品类"（category == "Computer"），你写成了 product == "Computer"（产品名）。UK 没有 product 叫 "Computer" 的，所以返回 Empty DataFrame。这是 #42 读题失误复发。

问题2（括号缺失）: df['total'] > 2000 在 & 表达式中没有加括号。虽然 > 优先级高于 & 碰巧对了，但这是隐患，一旦优先级搞混就会出 bug。

In [41]:
df[(df['country'] == 'UK') & (df['category'] == 'Computer') & (df['total'] > 2000)]

,order_id,customer_id,product,category,quantity,price,order_date,country,total,unit_price
72,O1072,C001,Monitor,Computer,3,999,2024-02-22,UK,2997.0,999.0
99,O1099,C005,Monitor,Computer,5,1299,2024-03-13,UK,6495.0,1299.0
100,O1100,C007,Laptop,Computer,4,1299,2024-03-14,UK,5196.0,1299.0
122,O1122,C002,Monitor,Computer,4,1999,2024-03-30,UK,7996.0,1999.0
132,O1132,C001,Monitor,Computer,5,1299,2024-04-06,UK,6495.0,1299.0
145,O1145,C007,Laptop,Computer,4,1299,2024-04-16,UK,5196.0,1299.0
222,O1222,C006,Laptop,Computer,5,999,2024-06-11,UK,4995.0,999.0
223,O1223,C004,Monitor,Computer,3,1999,2024-06-12,UK,5997.0,1999.0
225,O1225,C002,Laptop,Computer,4,1299,2024-06-13,UK,5196.0,1299.0
267,O1267,C008,Laptop,Computer,4,1299,2024-07-14,UK,5196.0,1299.0


**10. 综合管道 —— 读取、清洗、分析、输出**

写一段完整代码，完成以下流程（不要封装成函数，直接写脚本）：

**阶段1 —— 读取**:
- 读取 `../data/sales.csv`

**阶段2 —— 清洗**:
- 把 `order_date` 转为 datetime，失败转为 NaT（`errors="coerce"`）
- 把 `quantity` 转为 int，失败转为 NaN（`pd.to_numeric` + `errors="coerce"`，再 `astype("Int64")`）
- 把 `total` 转为 float，失败转为 NaN
- 删除存在 NaN 的行（`dropna()`）

**阶段3 —— 分析**:
- 新增 `unit_price` = `total / quantity`
- 按 `country` 分组（用筛选/循环，不用 groupby），统计每个国家的：
  - 订单数
  - 总销售额（`total` 之和）
  - 平均订单额（`total` 均值）
  - 最大订单额
- 结果存在一个 dict 里：`{country: {"count": ..., "sum": ..., "mean": ..., "max": ...}}`

**阶段4 —— 输出**:
- 把上述 dict 转成 DataFrame（提示：`pd.DataFrame.from_dict(result, orient="index")`）
- 按 `sum` 降序排列
- 写入 `country_summary.csv`（`index=True` 保留 country 作为索引列）

**边界检查**: 在 cell 末尾打印清洗前后的行数，确认清洗没有过度删除数据

In [39]:
df = pd.read_csv('../data/sales.csv')

df['order_date'] = pd.to_datetime(df['order_date'], errors= 'coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors= 'coerce').astype(int)
df['total'] = pd.to_numeric(df['total'], errors= 'coerce').astype(float)
df = df.dropna()

df["unit_price"] = df["total"] / df["quantity"]
country_stats = {}
unique_countries = df["country"].unique()
for country in unique_countries:
    sub_df = df[df["country"] == country]
    total_series = sub_df["total"]
    
    count = len(sub_df)                
    sum_total = total_series.sum()     
    mean_total = total_series.mean()   
    max_total = total_series.max()    
    
    country_stats[country] = {
        "count": count,
        "sum": sum_total,
        "mean": mean_total,
        "max": max_total
    }


df_country = pd.DataFrame.from_dict(country_stats, orient='index')
df_country_sorted = df_country.sort_values('sum', ascending=False)
df_country_sorted.to_csv('country_summary.csv', index=False, encoding='utf-8')
print(df.shape)
print(df.country.shape)
print(df_country_sorted.shape)

(500, 10)
(500,)
(5, 4)


参考答案

问题: 题目明确要求 "index=True 保留 country 作为索引列"，你写成了 index=False。结果 CSV 里没有 country 列，只有数值列。

正确版:

In [42]:
df_country_sorted.to_csv('country_summary.csv', index=True, encoding='utf-8')